# LoRA Fine-Tuning — Brain Tumor MRI (3-Class Classification)

Full LoRA pipeline for Brain Tumor 3-class classification (meningioma, glioma, pituitary) on the Cheng Brain Tumor MRI benchmark.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "requests", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
except ImportError:
    CHECKPOINT_DIR = "outputs/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Working directory:    {os.getcwd()}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Configuration

Load benchmark configuration for Brain Tumor MRI and default LoRA PEFT hyperparameters.

In [ ]:
"""
Load Brain Tumor MRI dataset configuration and LoRA hyperparameters.
"""
from typing import Any
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/brain_tumor.yaml")
peft_cfg: dict[str, Any] = load_config("configs/peft/lora.yaml")
print("Brain Tumor Dataset Configuration:", cfg["dataset"])
print("LoRA Fine-Tuning Configuration:", peft_cfg["lora"])

## 2. Build Datasets

Construct training, validation, and test PyTorch dataloaders from `cheng-brain-tumor-mri-detection.ndjson`.

In [ ]:
"""
Download MRI images from NDJSON manifest (if needed) and build Brain Tumor DataLoaders (BS=32).
"""
from io import BytesIO
import json
from pathlib import Path
from PIL import Image
import requests
import torch
from tqdm.auto import tqdm
from src.data import build_brain_tumor_dataloaders, build_brain_tumor_transforms

NDJSON_PATH = Path("cheng-brain-tumor-mri-detection.ndjson")
IMAGE_ROOT = Path("data/brain_tumor/images")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

records = []
with open(NDJSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        if r.get("type") == "image" and r.get("url"):
            records.append(r)

for r in tqdm(records, desc="Checking/Downloading MRI images"):
    dest = IMAGE_ROOT / r["file"]
    if not dest.exists():
        resp = requests.get(r["url"], timeout=15)
        resp.raise_for_status()
        Image.open(BytesIO(resp.content)).convert("RGB").save(dest)

cfg["training"]["batch_size"] = 32
pre = cfg["preprocessing"]
train_tf, val_tf = build_brain_tumor_transforms(
    pre["image_size"], pre["normalize_mean"], pre["normalize_std"],
    low_pct=cfg["dataset"].get("mri_clip_percentile_low", 5.0),
    high_pct=cfg["dataset"].get("mri_clip_percentile_high", 95.0),
)
train_loader, val_loader, test_loader = build_brain_tumor_dataloaders(
    cfg, NDJSON_PATH, image_root=IMAGE_ROOT,
    train_transforms=train_tf, val_transforms=val_tf,
)

# Inverse-frequency class weights to address Meningioma minority recall drop
counts = torch.zeros(3, dtype=torch.float64)
for b in train_loader:
    for lbl in b["labels"]:
        counts[int(lbl)] += 1
CLASS_WEIGHTS = (counts.sum() / (3.0 * counts.clamp(min=1.0))).float()
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
print("Class weights [Meningioma, Glioma, Pituitary]:", [round(float(w), 3) for w in CLASS_WEIGHTS])

## 3. Linear Probing Baseline

Train a frozen DINOv2 backbone paired with a trainable 3-class classification head (`MultiClassHead(768, 3)`).

In [ ]:
"""
Train and evaluate linear probing baseline on Brain Tumor MRI (with class weights & AUROC tracking).
"""
import copy
from typing import Any
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from src.evaluation.metrics import compute_multiclass_metrics
from src.models import count_trainable_parameters, freeze_backbone, load_backbone, MultiClassHead
from src.training.losses import MultiClassCELoss
from src.training.schedulers import get_cosine_schedule_with_warmup
from src.utils.logging_utils import ExperimentLogger

CLASS_NAMES = ["meningioma", "glioma", "pituitary"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def eval_bt(model: nn.Module, loader) -> dict[str, Any]:
    model.eval()
    logits_all, labels_all = [], []
    for batch in loader:
        pv = batch["pixel_values"].to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(pv)
        logits_all.append(logits.float().cpu().numpy())
        labels_all.append(batch["labels"].numpy())
    probs = torch.softmax(torch.from_numpy(np.concatenate(logits_all, axis=0)), dim=-1).numpy()
    return compute_multiclass_metrics(np.concatenate(labels_all, axis=0), probs, class_names=CLASS_NAMES)

backbone, _ = load_backbone()
freeze_backbone(backbone)
head_lp = MultiClassHead(in_features=768, num_classes=3)

class LinearProbeBrainTumor(nn.Module):
    def __init__(self, b: nn.Module, h: nn.Module) -> None:
        super().__init__()
        self.b = b
        self.h = h
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.b(pixel_values=pixel_values)
        return self.h(outputs.last_hidden_state[:, 0, :])

model_lp = LinearProbeBrainTumor(backbone, head_lp).to(device)
trainable_lp, total_lp = count_trainable_parameters(model_lp)
print(f"Linear Probe Trainable: {trainable_lp:,} / {total_lp:,} ({100 * trainable_lp / total_lp:.4f}%)")

criterion_ce = MultiClassCELoss(weight=CLASS_WEIGHTS.to(device))
optimizer_lp = AdamW(head_lp.parameters(), lr=5e-3, weight_decay=1e-4)
epochs_lp = 15
sched_lp = get_cosine_schedule_with_warmup(optimizer_lp, int(0.1 * epochs_lp * len(train_loader)), epochs_lp * len(train_loader))
scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

best_v, best_st = -1.0, None
for epoch in range(epochs_lp):
    model_lp.train()
    running_loss = 0.0
    for batch in train_loader:
        pix, lab = batch["pixel_values"].to(device, non_blocking=True), batch["labels"].to(device, non_blocking=True)
        optimizer_lp.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion_ce(model_lp(pix), lab)
        scaler.scale(loss).backward()
        scaler.step(optimizer_lp)
        scaler.update()
        sched_lp.step()
        running_loss += loss.item()
    vm = eval_bt(model_lp, val_loader)
    if vm["mean_auroc"] > best_v:
        best_v = vm["mean_auroc"]
        best_st = copy.deepcopy(model_lp.state_dict())
    print(f"LP Epoch [{epoch + 1:02d}/{epochs_lp}] Loss: {running_loss / len(train_loader):.4f} | Val AUROC: {vm['mean_auroc']:.4f} | Val Acc: {vm['accuracy']:.4f}")

model_lp.load_state_dict(best_st)
results_lp_val = eval_bt(model_lp, val_loader)
results_lp_test = eval_bt(model_lp, test_loader)
print(f"LP Best Val -> AUROC: {results_lp_val['mean_auroc']:.4f} | Acc: {results_lp_val['accuracy']:.4f} | F1: {results_lp_val['macro_f1']:.4f}")
print(f"LP Test     -> AUROC: {results_lp_test['mean_auroc']:.4f} | Acc: {results_lp_test['accuracy']:.4f} | F1: {results_lp_test['macro_f1']:.4f}")

logger = ExperimentLogger()
logger.log(
    run_id="lp_brain_tumor",
    dataset="BrainTumorMRI",
    backbone="DINOv2 ViT-B/14",
    method="LinearProbing",
    val_auroc=results_lp_val["mean_auroc"],
    test_auroc=results_lp_test["mean_auroc"],
    val_accuracy=results_lp_val["accuracy"],
    val_macro_f1=results_lp_val["macro_f1"],
    test_accuracy=results_lp_test["accuracy"],
    test_macro_f1=results_lp_test["macro_f1"],
)

## 4. LoRA Fine-Tuning (r=48)

Train DINOv2 ViT-B/14 adapted with standard LoRA (r=48) using `MultiClassCELoss`.

In [ ]:
"""
Train standard LoRA model (r=48) on Brain Tumor MRI with AMP, Cosine LR, class weights, and best-val checkpointing.
"""
from src.models import apply_lora

def build_bt_lora(r: int = 48, alpha: int = 16) -> LinearProbeBrainTumor:
    bb, _ = load_backbone()
    linear_names = {n.rsplit(".", 1)[-1] for n, m in bb.named_modules() if n and isinstance(m, nn.Linear)}
    attn_map = {"query": "q_proj", "value": "v_proj"}
    targets = [t if t in linear_names else attn_map.get(t, t) for t in ["query", "value"]]
    bb = apply_lora(bb, {"r": r, "lora_alpha": alpha, "target_modules": targets})
    return LinearProbeBrainTumor(bb, MultiClassHead(in_features=768, num_classes=3)).to(device)

model_lora = build_bt_lora(r=48, alpha=16)
trainable_lora, total_lora = count_trainable_parameters(model_lora)
print(f"LoRA Trainable: {trainable_lora:,} / {total_lora:,} ({100 * trainable_lora / total_lora:.2f}%)")

epochs_lora = 15
optimizer_lora = AdamW(
    [
        {"params": [p for p in model_lora.b.parameters() if p.requires_grad], "lr": 2e-4},
        {"params": model_lora.h.parameters(), "lr": 5e-3},
    ],
    weight_decay=1e-4,
)
sched_lora = get_cosine_schedule_with_warmup(optimizer_lora, int(0.1 * epochs_lora * len(train_loader)), epochs_lora * len(train_loader))
sc_lora = torch.amp.GradScaler(enabled=(device.type == "cuda"))

best_v, best_st = -1.0, None
for epoch in range(epochs_lora):
    model_lora.train()
    running_loss = 0.0
    for batch in train_loader:
        pix, lab = batch["pixel_values"].to(device, non_blocking=True), batch["labels"].to(device, non_blocking=True)
        optimizer_lora.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion_ce(model_lora(pix), lab)
        sc_lora.scale(loss).backward()
        sc_lora.unscale_(optimizer_lora)
        torch.nn.utils.clip_grad_norm_(model_lora.parameters(), 1.0)
        sc_lora.step(optimizer_lora)
        sc_lora.update()
        sched_lora.step()
        running_loss += loss.item()
    vm = eval_bt(model_lora, val_loader)
    if vm["mean_auroc"] > best_v:
        best_v = vm["mean_auroc"]
        best_st = copy.deepcopy(model_lora.state_dict())
    print(f"LoRA Epoch [{epoch + 1:02d}/{epochs_lora}] Loss: {running_loss / len(train_loader):.4f} | Val AUROC: {vm['mean_auroc']:.4f} | Val Acc: {vm['accuracy']:.4f}")

model_lora.load_state_dict(best_st)
results_lora_val = eval_bt(model_lora, val_loader)
results_lora_test = eval_bt(model_lora, test_loader)
print(f"LoRA (r=48) Val  -> AUROC: {results_lora_val['mean_auroc']:.4f} | Acc: {results_lora_val['accuracy']:.4f} | F1: {results_lora_val['macro_f1']:.4f}")
print(f"LoRA (r=48) Test -> AUROC: {results_lora_test['mean_auroc']:.4f} | Acc: {results_lora_test['accuracy']:.4f} | F1: {results_lora_test['macro_f1']:.4f}")

logger.log(
    run_id="lora_r48_brain_tumor",
    dataset="BrainTumorMRI",
    backbone="DINOv2 ViT-B/14",
    method="LoRA",
    rank=48,
    trainable_params=trainable_lora,
    val_auroc=results_lora_val["mean_auroc"],
    test_auroc=results_lora_test["mean_auroc"],
    val_accuracy=results_lora_val["accuracy"],
    val_macro_f1=results_lora_val["macro_f1"],
    test_accuracy=results_lora_test["accuracy"],
    test_macro_f1=results_lora_test["macro_f1"],
)

## 5. Rank Sweep [4, 8, 16, 32, 48]

Run rank sweep across ranks [4, 8, 16, 32, 48] on Brain Tumor MRI to observe multi-class capacity requirements.

In [ ]:
"""
Execute LoRA rank sweep over [4, 8, 16, 32, 48] on Brain Tumor MRI with best-val checkpoint selection.
"""
import pandas as pd

RANK_SWEEP_BT: list[int] = [4, 8, 16, 32, 48]
bt_sweep_records: list[dict[str, Any]] = []

for r in RANK_SWEEP_BT:
    print(f"\n=== Running Brain Tumor Rank Sweep: r={r} ===")
    m_r = build_bt_lora(r=r, alpha=16)
    tr_params, _ = count_trainable_parameters(m_r)
    opt_r = AdamW(
        [
            {"params": [p for p in m_r.b.parameters() if p.requires_grad], "lr": 2e-4},
            {"params": m_r.h.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )
    ep = 12
    sched_r = get_cosine_schedule_with_warmup(opt_r, int(0.1 * ep * len(train_loader)), ep * len(train_loader))
    sc_r = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    best_v, best_st = -1.0, None
    for epoch in range(ep):
        m_r.train()
        for batch in train_loader:
            pix, lab = batch["pixel_values"].to(device, non_blocking=True), batch["labels"].to(device, non_blocking=True)
            opt_r.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                loss = criterion_ce(m_r(pix), lab)
            sc_r.scale(loss).backward()
            sc_r.unscale_(opt_r)
            torch.nn.utils.clip_grad_norm_(m_r.parameters(), 1.0)
            sc_r.step(opt_r)
            sc_r.update()
            sched_r.step()
        vm = eval_bt(m_r, val_loader)
        if vm["mean_auroc"] > best_v:
            best_v = vm["mean_auroc"]
            best_st = copy.deepcopy(m_r.state_dict())

    m_r.load_state_dict(best_st)
    test_metrics = eval_bt(m_r, test_loader)
    print(f"Rank r={r} Test -> AUROC: {test_metrics['mean_auroc']:.4f} | Acc: {test_metrics['accuracy']:.4f} | F1: {test_metrics['macro_f1']:.4f}")
    bt_sweep_records.append(
        {"rank": r, "trainable_params": tr_params, "test_auroc": test_metrics["mean_auroc"], "accuracy": test_metrics["accuracy"], "macro_f1": test_metrics["macro_f1"]}
    )
    logger.log(
        run_id=f"lora_r{r}_brain_tumor",
        dataset="BrainTumorMRI",
        backbone="DINOv2 ViT-B/14",
        method="LoRA",
        rank=r,
        trainable_params=tr_params,
        test_auroc=test_metrics["mean_auroc"],
        test_accuracy=test_metrics["accuracy"],
        test_macro_f1=test_metrics["macro_f1"],
    )

bt_df = pd.DataFrame(bt_sweep_records)
print(bt_df.to_string(index=False))

## 6. Results and Confusion Matrix

Plot the test set confusion matrix and output a detailed classification report with precision, recall, and F1 scores per tumor class.

In [ ]:
"""
Generate confusion matrix heatmap and classification report table on held-out test split.
"""
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model_lora.eval()
all_preds: list[int] = []
all_targets: list[int] = []

with torch.no_grad():
    for batch in test_loader:
        pix = batch["pixel_values"].to(device)
        labs = batch["labels"]
        logits = model_lora(pix)
        preds = torch.argmax(logits, dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_targets.extend(labs.tolist())

class_names: list[str] = ["meningioma", "glioma", "pituitary"]
cm = confusion_matrix(all_targets, all_preds)
print("Classification Report:\n", classification_report(all_targets, all_preds, target_names=class_names))

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title("Brain Tumor LoRA Test Confusion Matrix", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Tumor Class", fontsize=11)
ax.set_ylabel("Ground Truth Class", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Log All Experiments

Review the updated centralized experiment log containing all Brain Tumor MRI runs.

In [ ]:
"""
Display full experiment logger dataframe to verify Brain Tumor MRI records.
"""
log_df = logger.to_dataframe()
print(log_df[log_df["dataset"] == "BrainTumorMRI"][["run_id", "method", "test_accuracy", "test_macro_f1"]])